# LA 03 — Trade-print-driven exit and structural target research

**Exploratory only; neither an MNQ-equivalent validated strategy nor prospective evidence.** Uses the exact **452 price-defined A02 candidates**, joined to A02b pre-signal flow. First actual print after the second 1-minute candle close is a proxy for entry. Three *identical-entry* policies are evaluated: **FIXED**, **BREAKEVEN**, **TRAIL_1M**. We simulate 1m *confirmed* swing structure, but **never infer stop/target ordering from minute OHLC**.

**Material assumptions requiring owner review:** BTC/ETH stop one A02 zone halfwidth beyond level (not MNQ 15–20 ticks), BE at +1 initial R (not MNQ +10 points), hypothetical 2-tick crypto trailing buffer, 240m timeout, 2R-displaced VWAP else opposing completed prior-day extreme; **no LVN targets**. Fills are based on trade prints with **assumed** adverse execution and fees. **Funding omitted.** See `discussions/Liquidity_Absorption/007_chatgpt_a03_execution_research_protocol.md` and the generated `03_execution_contract.json`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / "src" / "data_engineering").is_dir():
    if REPO == REPO.parent:
        raise RuntimeError("Launch Jupyter inside repository or a subdirectory")
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a03 import Config, VERSION, run_a03
assert VERSION == "LA-A03-0.1.0", "Pull repository and restart kernel"
DATA_ROOT = binance_root(REPO)
OUT = REPO / "notebooks" / "Liquidity_Absorption" / "03"
CFG = Config()
print("A03:", VERSION, "| Binance root:", DATA_ROOT, "| Output:", OUT)
print("EXPLORATORY historical prints, assumed fills & costs; funding excluded; NO HOLDOUT")

A03: LA-A03-0.1.0 | Binance root: D:\MarketData\Binance | Output: D:\Projects\deribit_market_data\notebooks\Liquidity_Absorption\03
EXPLORATORY historical prints, assumed fills & costs; funding excluded; NO HOLDOUT


## Run three policies over historical trade prints

Reads only the existing daily aggTrade Parquet files in bounded batches (no downloaded data or duplicate tick archive). The source event population remains unchanged. This can be CPU-intensive because all print crossings must be ordered. If the final day lacks post-event tape, those positions must be marked censored—not marked as losses or winners.

In [2]:
decision = run_a03(REPO, DATA_ROOT, OUT, CFG)
print({k: decision[k] for k in ("version", "status", "source_events", "simulated_policy_rows", "tape")})
display(pd.DataFrame(decision["coverage"]))

{'version': 'LA-A03-0.1.0', 'status': 'EXPLORATORY_EXECUTION_SIMULATION_NOT_CONFIRMED_STRATEGY', 'source_events': 452, 'simulated_policy_rows': 1356, 'tape': {'read_daily_parquet_files': 171, 'missing_trade_day_files': ['BTCUSDT:2026-10-01', 'ETHUSDT:2026-10-01']}}


,symbol,mode,events,closed_trades,no_entry_or_no_target,censored,no_entry_tape,stop_exits,target_exits,timeouts,flow_quality_flagged,mean_net_r_descriptive,median_net_r_descriptive,positive_net_r_share_descriptive,mean_gross_r_descriptive
0,BTCUSDT,BREAKEVEN,212,196,16,0,0,173,20,3,1,-2.143479,-2.742590,0.102041,0.181714
1,BTCUSDT,FIXED,212,196,16,0,0,166,24,6,1,-2.253105,-3.254844,0.137755,0.066114
2,BTCUSDT,TRAIL_1M,212,196,16,0,0,181,15,0,1,-2.025091,-2.517068,0.096939,0.306069
3,ETHUSDT,BREAKEVEN,240,219,21,0,0,205,13,1,2,-2.254487,-2.686581,0.063927,-0.058451
4,ETHUSDT,FIXED,240,219,21,0,0,191,21,7,2,-2.249299,-3.056521,0.123288,-0.061895
5,ETHUSDT,TRAIL_1M,240,219,21,0,0,212,7,0,2,-2.021850,-2.544616,0.127854,0.178337


In [3]:
events = pd.read_csv(OUT / "03_execution_events.csv")
coverage = pd.read_csv(OUT / "03_execution_coverage.csv")
assert len(events) == 452 * 3
assert events.groupby("a02_candidate_row").size().eq(3).all()
assert set(events["mode"]) == {"FIXED", "BREAKEVEN", "TRAIL_1M"}
assert not events["reason"].isna().any()
print("Exit reasons across policies:")
display(events.groupby(["symbol", "mode", "reason"]).size().rename("n").reset_index())
print("Provisional target-selection sources:")
display(events.loc[events["mode"]=="FIXED", "target_source"].value_counts(dropna=False).rename_axis("target_source").reset_index(name="events"))
print("Potential A02b flow-reconciliation outliers (candidate count):",
      events.loc[events["mode"]=="FIXED", "flow_integrity_flag"].value_counts(dropna=False).to_dict())
print("Contract explicitly excludes funding, executable bid/ask and LVN targets.")

Exit reasons across policies:


,symbol,mode,reason,n
0,BTCUSDT,BREAKEVEN,NO_STRUCTURAL_TARGET,16
1,BTCUSDT,BREAKEVEN,STOP,173
2,BTCUSDT,BREAKEVEN,TARGET,20
3,BTCUSDT,BREAKEVEN,TIMEOUT,3
4,BTCUSDT,FIXED,NO_STRUCTURAL_TARGET,16
5,BTCUSDT,FIXED,STOP,166
6,BTCUSDT,FIXED,TARGET,24
7,BTCUSDT,FIXED,TIMEOUT,6
8,BTCUSDT,TRAIL_1M,NO_STRUCTURAL_TARGET,16
9,BTCUSDT,TRAIL_1M,STOP,181


Provisional target-selection sources:


,target_source,events
0,SESSION_VWAP,233
1,OPPOSING_PRIOR_DAY,182
2,NOT_ENTERED,37


Potential A02b flow-reconciliation outliers (candidate count): {'OK': 449, 'RECONCILIATION_OUTLIER': 3}
Contract explicitly excludes funding, executable bid/ask and LVN targets.


## Interpretation gate (read before discussing P&L)

- The observed prints determine when a barrier was reached, **not** actual stop-market / bid-ask fills. Cost figures are **assumptions**, not measured economics; funding and market impact/queue uncertainty remain.
- A **BE stop is not zero net P&L**: entry and exit impact and fees are still charged.
- Three policies on one candidate are **correlated counterfactuals**, not independent portfolios. Censored and skipped events cannot be omitted from denominators without notice.
- The three-condition `absorption_like_exploratory` flag is a **post-hoc descriptive intersection** already inspected at A02b, not a validated or optimized signal.
- No holdout was reserved in July–September 2026. We must obtain owner approval for stop-distance, BE mapping, swing and target semantics—and reserve an untouched later period—before calling any finding predictive or tradable.

Commit/push the executed notebook and the 4 generated files in `notebooks/Liquidity_Absorption/03/`; then request results review.